# Notebook 7 – Boosting

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity`, `UnitPrice`, `TotalPrice`.

## Setup: Load & Split Data

In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(3000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. What is Boosting?
An ensemble technique that builds models **one after another**, where each new model tries to correct the mistakes made by the previous ones — rather than training all models independently like bagging does.

In [2]:
from sklearn.ensemble import AdaBoostClassifier
boost = AdaBoostClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)
print("Boosting test acc:", accuracy_score(y_test, boost.predict(X_test)))

Boosting test acc: 0.8966666666666666


## 2. Sequential Learning
Unlike bagging (where models train independently and in parallel), boosting trains models **sequentially** — each step depends on the errors of the step before it.

In [3]:
print("Number of sequential models trained:", len(boost.estimators_))
print("First few model weights:", boost.estimator_weights_[:5])

Number of sequential models trained: 50
First few model weights: [2.16072418 0.67383834 0.38972329 0.42768479 0.1958833 ]


## 3. Weak Learners
Boosting typically combines many **weak learners** — models that are only slightly better than random guessing on their own (e.g., a very shallow Decision Tree, sometimes called a "stump").

In [4]:
from sklearn.tree import DecisionTreeClassifier
weak_learner = DecisionTreeClassifier(max_depth=1, random_state=42).fit(X_train, y_train)
print("A single weak learner (stump) test acc:", accuracy_score(y_test, weak_learner.predict(X_test)))

A single weak learner (stump) test acc: 0.8966666666666666


## 4. Strong Learner
By combining many weak learners sequentially, boosting builds a **strong learner** — a combined model that performs much better than any individual weak learner alone.

In [5]:
print("Weak learner alone:", accuracy_score(y_test, weak_learner.predict(X_test)))
print("Boosted strong learner:", accuracy_score(y_test, boost.predict(X_test)))

Weak learner alone: 0.8966666666666666
Boosted strong learner: 0.8966666666666666


## 5. Gradient Boosting
Builds models sequentially, where each new model is trained to predict the **residual errors** (gradient of the loss) of the combined model so far — a more general and often more powerful approach than AdaBoost.

In [7]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
print("Gradient Boosting test acc:", accuracy_score(y_test, gb.predict(X_test)))

Gradient Boosting test acc: 0.9


## 6. AdaBoost
Short for **Adaptive Boosting**. After each round, it increases the weight of misclassified examples so the next weak learner focuses more on the hard cases.

In [8]:
print("AdaBoost test acc:", accuracy_score(y_test, boost.predict(X_test)))

AdaBoost test acc: 0.8966666666666666


## 7. HistGradientBoosting
A faster, histogram-based version of Gradient Boosting (inspired by LightGBM). It bins continuous features into discrete buckets before finding splits, making it much faster on larger datasets.

In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier
hgb = HistGradientBoostingClassifier(random_state=42).fit(X_train, y_train)
print("HistGradientBoosting test acc:", accuracy_score(y_test, hgb.predict(X_test)))

HistGradientBoosting test acc: 0.8766666666666667


## Comparison: All Boosting Methods

In [11]:
results = pd.DataFrame({
    'Model': ['Weak Stump', 'AdaBoost', 'Gradient Boosting', 'HistGradientBoosting'],
    'Test Accuracy': [
        accuracy_score(y_test, weak_learner.predict(X_test)),
        accuracy_score(y_test, boost.predict(X_test)),
        accuracy_score(y_test, gb.predict(X_test)),
        accuracy_score(y_test, hgb.predict(X_test)),
    ]
})
results

,Model,Test Accuracy
0,Weak Stump,0.896667
1,AdaBoost,0.896667
2,Gradient Boosting,0.900000
3,HistGradientBoosting,0.876667


## Bagging vs Boosting — Key Difference

| | **Bagging** | **Boosting** |
|---|---|---|
| Training | Models trained **independently, in parallel** | Models trained **sequentially**, each fixing previous errors |
| Data used per model | Random bootstrap samples | Full data, but reweighted / residual-focused each round |
| Main goal | Reduce **variance** (overfitting) | Reduce **bias** (underfitting) |
| Risk | Less prone to overfitting | Can overfit if too many rounds / too complex |
| Example | Random Forest | AdaBoost, Gradient Boosting, XGBoost |

**In short:** Bagging builds many models independently and averages them out to be more stable; Boosting builds models one at a time, each specifically targeting the previous model's mistakes, to push accuracy higher.